# NB-10-03 — Deep Reinforcement Learning

**Module M10 · Reinforcement Learning** · [L10-04](../lessons/L10-04-deep-reinforcement-learning.md)

Replace the table with a network and see the two stabilisers in action. The budgets here are deliberately
short: this notebook demonstrates the machinery and the shape of the curves, not a solved benchmark. Read
the numbers as "the pipeline works", never as "the task is solved".

In [1]:
import sys
sys.path.insert(0, "../code")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rl_envs, dp, tabular, experiments

print("experiments self-test:", experiments.run_tests())

experiments self-test: True


## 1. CartPole in one cell

Gymnasium provides the environment. If it cannot be created, the module's `RobotNavEnv` has the same
interface — record which one you used.

In [2]:
import gymnasium as gym

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("observation shape:", obs.shape, "actions:", env.action_space.n)
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("one random step:", np.round(obs, 3), reward, terminated, truncated)
env.close()

observation shape: (4,) actions: 2
one random step: [ 0.013 -0.217 -0.047  0.23 ] 1.0 False False


## 2. A short DQN run

Sixty episodes is enough to see whether the pipeline learns at all. The target network and replay buffer are
inside `dqn.train_dqn`; the next sections remove them.

In [3]:
import dqn

result = dqn.train_dqn(env_id="CartPole-v1", episodes=60, seed=0)
returns = np.asarray(result["returns"])
print("episodes:", len(returns))
print("first 10 mean:", round(float(returns[:10].mean()), 1))
print("last 10 mean: ", round(float(returns[-10:].mean()), 1))
print("smoothed tail:", np.round(dqn.smooth(returns, 20)[-5:], 1))
print("epsilon at 0, 2000 steps:", round(dqn.epsilon_schedule(0), 2), round(dqn.epsilon_schedule(2000), 2))

episodes: 60
first 10 mean: 21.7
last 10 mean:  16.0
smoothed tail: [17.2 16.5 16.6 15.8 15.3]
epsilon at 0, 2000 steps: 1.0 0.62


## 3. The network and the buffer

Both are ordinary PyTorch objects. Check the parameter count and that a minibatch has the expected shapes —
most DQN bugs are shape bugs.

In [4]:
net = dqn.QNet(4, 2)
print("parameters:", sum(p.numel() for p in net.parameters()))
print("output shape:", tuple(net(__import__("torch").zeros(1, 4)).shape))

buffer = dqn.ReplayBuffer(capacity=500)
for i in range(50):
    buffer.add(np.zeros(4, dtype=np.float32), 0, 1.0, np.ones(4, dtype=np.float32), False)
print("buffer size:", buffer.size)

parameters: 4610
output shape: (1, 2)
buffer size: 50


## 4. Policy gradient on the same task

REINFORCE optimises the policy directly. Short budgets mean noisy curves; the comparison is about variance
and sample efficiency, not about who "wins".

In [5]:
pg = dqn.reinforce(env_id="CartPole-v1", episodes=40, seed=0)
pg_returns = np.asarray(pg["returns"])
print("REINFORCE first 10:", round(float(pg_returns[:10].mean()), 1),
      "last 10:", round(float(pg_returns[-10:].mean()), 1))
print("REINFORCE std (last 10):", round(float(pg_returns[-10:].std()), 1))

REINFORCE first 10: 23.2 last 10: 23.4
REINFORCE std (last 10): 7.2


## 5. What would make this credible

A solve claim needs hundreds of episodes, three or more seeds, a scripted baseline and a stated threshold.
Write the protocol before running anything — it is the only defence against tuning until the picture looks
good.

In [6]:
protocol = {
    "episodes": ">= 400 per seed",
    "seeds": [0, 1, 2],
    "metric": "mean return over last 50 episodes",
    "threshold": 195.0,
    "baselines": ["uniform random", "scripted angle policy"],
    "report": "median across seeds with min/max",
}
for key, value in protocol.items():
    print(f"{key:10s}: {value}")

episodes  : >= 400 per seed
seeds     : [0, 1, 2]
metric    : mean return over last 50 episodes
threshold : 195.0
baselines : ['uniform random', 'scripted angle policy']
report    : median across seeds with min/max


## 6. Exercise

1. Run DQN for 200 episodes and compare the tail with the 60-episode run above.
2. Remove the target network in a copy of the loop and describe the curve — this is the ablation the lab
   asks for.
3. Run REINFORCE with a learned baseline (`dqn.actor_critic`) on the same budget and compare variance.

## Self-assessment

- Why does replay decorrelate samples?
- What does the target network hold still, and why does that help?
- Why is a single short run not evidence of learning?